# Entrega 02 · Preparación de datos de Isaac

El objetivo de esta notebook es convertir una exportación real de **Isaac Run History** en una tabla con una fila por piso. El piloto se centra en **Isaac en Hard** y en el objetivo de derrotar a **Mom’s Heart / It Lives** durante la run.

**Para ejecutar:** abre la notebook con un kernel de Python que tenga `pandas` instalado y pulsa **Run All**. Coloca la exportación actualizada del mod en `data/raw/save1.dat`. La salida se escribe en `data/gold/isaac_floors.csv`.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

## 1. Bronze · Leer la exportación original

El archivo del mod usa extensión `.dat` y contiene JSON. `runs` contiene el historial registrado y `current` describe la partida en curso. `lifetime` guarda el progreso acumulado del jugador. Para construir el dataset se lee únicamente el historial de runs.

In [2]:
# VS Code puede iniciar el kernel en la raíz del proyecto o en notebooks/.
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

raw_path = project_root / "data" / "raw" / "save1.dat"
gold_path = project_root / "data" / "gold" / "isaac_floors.csv"

if not raw_path.is_file():
    raise FileNotFoundError(f"Coloca la exportación de Isaac Run History en {raw_path}")

export = json.loads(raw_path.read_text(encoding="utf-8-sig"))
if export.get("format") != "isaac-run-history" or not isinstance(export.get("runs"), list):
    raise ValueError("El archivo debe ser una exportación JSON de Isaac Run History.")

runs = export["runs"]
print(f"Versión del mod: {export.get('modVersion', 'sin indicar')}")
print(f"Runs en el historial: {len(runs)}")
print(f"Run en curso: {'sí' if export.get('current') else 'no'}")

Versión del mod: 1.4.5
Runs en el historial: 1
Run en curso: sí


## 2. Silver · Preparar los pisos

Me quedo con Isaac en Hard. Las runs duplicadas se identifican por su ID y los pisos se enumeran en el orden en el que se jugaron. La estadística de cada fila sale de `floor["stats"]`, que corresponde al momento de entrada. La salud y las monedas quedan vacías porque el export del piloto no las registra en ese momento.

Para localizar el objetivo uso los IDs de jefe 8 y 25. Si una run solo trae una marca en `completions` o en el resultado final y no permite localizar el piso en que se consiguió, la dejo fuera de este Gold. Así evito incluir pisos posteriores al objetivo.

[Esquema de Isaac Run History](https://github.com/tomandesMSH/isaac-run-history/blob/main/schema/run-history.schema.json) · [IDs de jefes de REPENTOGON](https://repentogon.com/enums/BossType.html)

In [3]:
TARGET_BOSS_IDS = {8, 25}  # Mom's Heart e It Lives (BossType).
TARGET_EVENTS = {"MOMS_HEART", "IT_LIVES"}
FRAMES_PER_SECOND = 30

stat_fields = {
    "attack_damage": "damage",
    "fire_rate": "tears",
    "attack_range": "range",
    "move_speed": "speed",
    "shot_speed": "shotSpeed",
    "luck": "luck",
}

rows = []
seen_run_ids = set()
omitted_outcomes = 0

for run in runs:
    character = run.get("character") or {}
    if character.get("id") != 0 or run.get("difficulty") != "hard" or run.get("partial"):
        continue

    run_id = run.get("id")
    if not run_id or run_id in seen_run_ids:
        continue
    seen_run_ids.add(run_id)

    floors = run.get("floors") or []
    result = run.get("result") or {}
    result_type = result.get("type")
    # La secuencia del export distingue incluso pisos con el mismo stage.
    target_floor_seq = next(
        (
            floor_seq
            for floor_seq, floor in enumerate(floors, start=1)
            if any(boss.get("bossId") in TARGET_BOSS_IDS for boss in floor.get("bosses", []))
        ),
        None,
    )

    target_reported = any(mark in TARGET_EVENTS for mark in run.get("completions", []))
    target_reported |= result.get("ending") in TARGET_EVENTS

    # Una muerte cuenta como 0 si no hay evidencia de haber alcanzado el objetivo.
    # Para etiquetar un éxito por piso necesitamos localizar al jefe en el registro.
    if result_type not in {"death", "win"} or (
        target_floor_seq is None and (target_reported or result_type != "death")
    ):
        omitted_outcomes += 1
        continue

    for floor_seq, floor in enumerate(floors, start=1):
        if target_floor_seq is not None and floor_seq > target_floor_seq:
            break  # El objetivo ya se había conseguido al entrar en estos pisos.

        stats = floor.get("stats") or {}
        rows.append({
            "run_id": run_id,
            "character": character.get("name", "Isaac"),
            "difficulty": run["difficulty"],
            "floor_seq": floor_seq,
            "stage": floor.get("stage"),
            "stage_type": floor.get("stageType"),
            "entered_at_frame": floor.get("enteredAtFrame"),
            **{column: stats.get(source) for column, source in stat_fields.items()},
            "health_half_hearts": pd.NA,
            "coins": pd.NA,
            "target_mark": "MOMS_HEART",
            "reached_target_after_snapshot": int(target_floor_seq is not None),
        })

if not rows:
    raise ValueError(
        "El historial no contiene pisos etiquetables de Isaac en Hard. "
        "Comprueba que data/raw/save1.dat sea el archivo guardado después de terminar la run."
    )

silver = pd.DataFrame(rows)
integer_columns = ["stage", "stage_type", "entered_at_frame", "health_half_hearts", "coins"]
silver[integer_columns] = silver[integer_columns].astype("Int64")
silver[list(stat_fields)] = silver[list(stat_fields)].astype("Float64")
print(f"Runs incluidas: {silver['run_id'].nunique()} · Pisos incluidos: {len(silver)}")
print(f"Runs del piloto omitidas por resultado no etiquetable: {omitted_outcomes}")

Runs incluidas: 1 · Pisos incluidos: 7
Runs del piloto omitidas por resultado no etiquetable: 0


## 3. Gold · Validar y exportar

Cada fila representa el estado al entrar en un piso y se identifica por `run_id + floor_seq + target_mark`. La etiqueta vale **1** si posteriormente se derrota al jefe objetivo y **0** si la run termina en muerte sin derrotarlo. En la muestra de la entrega esperamos **una run con siete pisos y siete etiquetas 0**, porque la muerte fue intencionada para probar el flujo.

In [4]:
gold = silver.assign(elapsed_seconds=(silver["entered_at_frame"] / FRAMES_PER_SECOND).round(2))
gold = gold[[
    "run_id", "character", "difficulty", "floor_seq", "stage", "stage_type",
    "elapsed_seconds", *stat_fields, "health_half_hearts", "coins",
    "target_mark", "reached_target_after_snapshot",
]].sort_values(["run_id", "floor_seq"]).reset_index(drop=True)

key = ["run_id", "floor_seq", "target_mark"]
if gold.duplicated(key).any():
    raise ValueError("El Gold contiene filas duplicadas para una run, piso y objetivo.")
if not gold["reached_target_after_snapshot"].isin([0, 1]).all():
    raise ValueError("Se esperaba una etiqueta 0 o 1 en todas las filas.")

gold_path.parent.mkdir(parents=True, exist_ok=True)
gold.to_csv(gold_path, index=False, encoding="utf-8")

display(gold)
print(f"Gold guardado en {gold_path.relative_to(project_root)}")

,run_id,character,difficulty,floor_seq,stage,stage_type,elapsed_seconds,attack_damage,fire_rate,attack_range,move_speed,shot_speed,luck,health_half_hearts,coins,target_mark,reached_target_after_snapshot
0,2203360706-1795172157-0,Isaac,hard,1,1,0,0.0,3.5,2.73,6.5,1.0,1.0,0.0,<NA>,<NA>,MOMS_HEART,0
1,2203360706-1795172157-0,Isaac,hard,2,2,2,145.8,4.43,2.73,8.0,1.0,1.0,0.0,<NA>,<NA>,MOMS_HEART,0
2,2203360706-1795172157-0,Isaac,hard,3,3,1,343.03,4.43,3.45,10.5,1.0,1.0,0.0,<NA>,<NA>,MOMS_HEART,0
3,2203360706-1795172157-0,Isaac,hard,4,4,0,651.63,4.43,4.13,10.5,1.0,1.0,0.0,<NA>,<NA>,MOMS_HEART,0
4,2203360706-1795172157-0,Isaac,hard,5,5,2,919.4,18.14,4.13,10.5,1.0,1.0,0.0,<NA>,<NA>,MOMS_HEART,0
5,2203360706-1795172157-0,Isaac,hard,6,6,2,1160.2,9.48,4.13,10.5,1.3,1.0,0.0,<NA>,<NA>,MOMS_HEART,0
6,2203360706-1795172157-0,Isaac,hard,7,7,1,1436.27,6.22,4.13,10.5,1.3,1.0,0.0,<NA>,<NA>,MOMS_HEART,0


Gold guardado en data\gold\isaac_floors.csv
